# **Scenario 1: Design a Distributed Job Scheduling and Execution Engine**

**The Context:**
You are building an internal "Task Execution Platform" for a fast-growing e-commerce company. Developers from various teams will submit background jobs to this system—anything from generating daily financial reports and processing large image batches, to sending out millions of promotional emails. 

**Initial Parameters:**
* **Scale:** The system receives roughly 10,000 job submissions per minute. 
* **Workload Variance:** Job execution times are highly variable. Some take 5 seconds; others take 2 hours. 
* **Isolation:** Since different teams write these Python jobs, they might have conflicting dependencies. A crash in one job must never take down the system or affect other running jobs.
* **Reliability:** Jobs cannot be lost if a server crashes. The system needs to guarantee at least *at-least-once* execution, with a preference for handling retries gracefully.
* **Operations (The DevOps Angle):** You need a strategy for auto-scaling the underlying compute resources based on queue depth, and a safe way to deploy updates to the worker nodes without killing currently processing, long-running jobs.

### 1. The Core Context: What is the System's Purpose?

**The Business Need:**
Imagine a large e-commerce company. The main website (the part customers see) needs to be fast and responsive. However, there are many heavy tasks that need to happen that would slow down the website if done immediately.
*   **Example:** When a user uploads a profile picture, the website shouldn't freeze while resizing that image. It should say "Upload Complete" and then handle the resizing in the background.
*   **Example:** At the end of the day, the finance team needs a report summarizing millions of transactions. This takes a long time and shouldn't block the website.

**Your System's Role:**
You are building the "background worker" system. Developers from other teams will send tasks to your system, and your system is responsible for finding a computer to run that task, running it, and reporting back when it's done. You are essentially building a **factory for code execution**.

**The Multi-Tenant Aspect:**
Many different teams (Finance, Marketing, Engineering) will use your system. They do not coordinate with each other. One team might write efficient code; another might write code that leaks memory. Your system must treat all of them fairly and safely.

---

### 2. Scale: The Ingestion Pressure

**Requirement:** `10,000 job submissions per minute`

**What this challenges you to solve:**
*   **The Entry Point:** Your system needs an "entry point" (an API) where developers send their jobs. This entry point must be able to accept 10,000 requests every minute without slowing down or rejecting them.
*   **Buffering:** Since jobs take time to process (seconds to hours), you cannot process them instantly as they arrive. You need a place to store these jobs temporarily while they wait for a worker to pick them up. This storage needs to be fast enough to handle the write speed of 10,000 items per minute.
*   **Backpressure:** What happens if jobs are coming in faster than they can be processed? The waiting line (queue) will grow. Your system needs to handle a growing line without crashing.

**Key Question for You:** How do you ensure the system accepts jobs quickly even if the processing side is slow?

---

### 3. Workload Variance: The Time Problem

**Requirement:** `Job execution times are highly variable. Some take 5 seconds; others take 2 hours.`

**What this challenges you to solve:**
*   **Resource Hoarding:** A job that takes 2 hours occupies a worker machine for a long time. If you have many 2-hour jobs running, you might not have enough workers left to handle the 5-second jobs.
*   **Efficiency:** Is it efficient to use the same type of worker machine for a 5-second task as you do for a 2-hour task? Maybe not.
*   **Visibility:** If a job runs for 2 hours, how do you know it's still alive? What if it got stuck after 1 hour? Your system needs a way to distinguish between "still working" and "dead/stuck."
*   **Queue Blocking:** If you have a single line for all jobs, a long job at the front of the line might delay thousands of short jobs behind it.

**Key Question for You:** How do you manage workers so that short jobs aren't delayed by long jobs, and how do you track the status of tasks that run for hours?

---

### 4. Isolation: The Safety Problem

**Requirement:** `Conflicting dependencies. A crash in one job must never take down the system or affect other running jobs.`

**What this challenges you to solve:**
*   **Library Conflicts:** Team A's job needs Version 1 of a software library. Team B's job needs Version 2. They cannot exist on the same standard environment simultaneously.
*   **Resource Contention:** If one job goes into an infinite loop and uses 100% of the CPU, it shouldn't slow down the other jobs running on the same physical machine.
*   **Memory Safety:** If one job leaks memory and fills up the RAM, it shouldn't cause the other jobs to crash due to lack of memory.
*   **Security:** One team's job should not be able to read the data or environment variables of another team's job.

**Key Question for You:** How do you run untrusted code from different teams on the same physical hardware without them interfering with each other?

---

### 5. Reliability: The Failure Problem

**Requirement:** `Jobs cannot be lost if a server crashes. At-least-once execution. Handle retries gracefully.`

**What this challenges you to solve:**
*   **Durability:** When a job is submitted, it must be saved permanently. If your entire system loses power immediately after submission, that job must still exist when the system turns back on.
*   **Worker Failure:** Workers (the machines running the code) will crash. It is a fact of distributed systems. If a worker crashes while running a job, that job needs to be picked up by a different worker.
*   **Double Execution:** Because workers crash, you might end up running a job twice (e.g., Worker A starts, crashes, Worker B picks it up). Your system must allow for this possibility. This means the code running inside the job needs to be safe to run multiple times (idempotent), or your system needs to prevent double execution.
*   **Retry Logic:** If a job fails because of a temporary glitch (like a network timeout), you should try again. If it fails because of a bug in the code, retrying won't help. You need a way to distinguish these and stop retrying eventually.

**Key Question for You:** How do you ensure a job is never lost, even if the machine running it dies, while acknowledging that a job might run more than once?

---

### 6. Operations: The Maintenance Problem

**Requirement:** `Auto-scaling based on queue depth. Safe way to deploy updates without killing long-running jobs.`

**What this challenges you to solve:**
*   **Cost vs. Performance:** You don't want to pay for 1,000 workers when there are no jobs. You don't want 10 workers when there are 1 million jobs. You need a mechanism to add/remove workers automatically based on how much work is waiting.
*   **Updating the System:** You will need to update the software that runs the workers.
    *   **The Risk:** If you restart all workers to update the software, you will kill any job that is currently running (especially those 2-hour jobs).
    *   **The Challenge:** You need a way to update the software on a worker only after it has finished its current task, not while it is in the middle of one.

**Key Question for You:** How do you add/remove capacity automatically, and how do you update the worker software without interrupting tasks that are already in progress?

---

### Summary of Tensions (Trade-offs to Consider)

As you design this, you will find that some requirements fight against each other. Here are the main tensions you need to balance:

1.  **Speed vs. Safety:** To make things fast, you might want to run many jobs on one machine. To make things safe (Isolation), you want to separate them, which uses more resources.
2.  **Efficiency vs. Reliability:** To be efficient, you want to run a job exactly once. To be reliable (in case of crashes), you must be prepared to run it at least once, which risks running it twice.
3.  **Utilization vs. Latency:** To save money, you want workers to be 100% busy. But if they are 100% busy, new jobs have to wait in line (increasing latency).
4.  **Deployment Speed vs. Job Continuity:** To deploy updates fast, you want to restart workers immediately. To protect long jobs, you need to wait for them to finish before restarting.

### Final Thought for Your Design

When you start drawing your diagram or writing your solution, keep this narrative in mind:
*"I need to accept a high volume of tasks, store them safely, distribute them to isolated environments that can run different types of code, ensure they finish even if machines break, and manage the fleet of machines efficiently without interrupting work."*

---

## Phase 1 — Clarifying Questions (Ask These First, Always)

In a real interview, before drawing a single box, you ask clarifying questions. This demonstrates senior thinking — you understand that requirements are ambiguous and assumptions are dangerous.

---

### Question 1 — Job Submission Interface
> "When developers submit a job, is this via a REST API call? Or are we integrating with existing code — like a Python decorator or a Celery-style `.delay()` call? This affects whether I need an SDK layer on top of the API."

**Candidate's Answer:**
Developers submit via a REST API. We'll also provide a lightweight Python SDK that wraps the API calls so they don't have to deal with raw HTTP. The SDK is a client concern — the core system is API-first.

---

### Question 2 — Job Definition Format
> "What does a job actually look like? Is it arbitrary code that gets uploaded? A Docker image reference? A pre-registered function name? This defines our execution model entirely."

**Candidate's Answer:**
Jobs are pre-registered Python functions, stored in versioned Docker images. A developer registers a job type (e.g., `generate_finance_report:v3`) and the system knows which image to pull. At submission time, they send the job type + input payload (JSON). We don't accept arbitrary code at runtime — images are built and pushed via CI/CD.

---

### Question 3 — Retry and Failure Semantics
> "When a job fails, who decides the retry policy — the platform or the developer? Is there a maximum retry count? Should failed jobs go to a dead-letter queue? And does the business care about *exactly-once* or is *at-least-once* acceptable?"

**Candidate's Answer:**
Developers declare retry policy at job registration (max retries, backoff strategy). The platform enforces it. After exhausting retries, jobs go to a dead-letter queue for manual inspection. The business accepts at-least-once execution — jobs should be written idempotently. Exactly-once is a bonus, not a hard requirement.

---

### Question 4 — Priority and SLA
> "Are all jobs equal? If a finance report and a marketing email batch are both queued, does one take precedence? Do we have SLA commitments — like '5-second jobs must start within 30 seconds of submission'?"

**Candidate's Answer:**
Yes — three priority tiers: HIGH (< 30s start SLA), NORMAL (best effort), LOW (background, can wait hours). Each tier gets its own queue. No strict SLA contracts for now, but we track P99 latency per tier for observability.

---

### Question 5 — Isolation Level
> "How strict is the isolation requirement? Process-level isolation (cheap, fast) or container-level isolation (stronger, more overhead)? If Team A's job can crash the Python interpreter, does that affect Team B?"

**Candidate's Answer:**
Container-level isolation is required. Each job runs in its own container — separate filesystem, network namespace, resource limits. Process-level isn't enough given conflicting dependencies and the multi-tenant security requirement.

---

### Question 6 — State and Output
> "Where do job results go? Does the submitter poll for results, or do we push via webhook/callback? And do we need to store job outputs long-term or just the status?"

**Candidate's Answer:**
Submitter gets a `job_id` at submission. They can poll a status endpoint. Results (output payload) are stored for 7 days. For async workflows, teams can register a webhook callback URL — we POST the result when the job completes. Both patterns must be supported.

---

### Question 7 — Operational Constraints
> "What cloud provider? Are we on Kubernetes? Do we have an existing message broker or do I propose one? And what's our budget sensitivity — cost-optimized or performance-first?"

**Candidate's Answer:**
Cloud-agnostic design, but assume AWS for specifics. We have Kubernetes (EKS). No existing broker — I can propose. Balance cost and performance — don't over-engineer, but don't cut corners on reliability.

---

## Phase 2 — Requirements Crystallization

### Functional Requirements

1. Accept job submissions via REST API (+ Python SDK)
2. Execute jobs as isolated containers (one job = one container)
3. Support three priority queues (HIGH / NORMAL / LOW)
4. Guarantee at-least-once execution with configurable retry + exponential backoff
5. Dead-letter queue for permanently failed jobs
6. Return `job_id` on submission; expose status polling endpoint
7. Support webhook callback on job completion
8. Store job results for 7 days

### Non-Functional Requirements

| Requirement | Target |
|---|---|
| Ingestion throughput | 10,000 submissions/minute (~167/sec) |
| Job execution latency (SHORT jobs) | < 30s from submission to start |
| Availability | 99.9% (API + scheduler) |
| Durability | Zero job loss — jobs must survive full server crash |
| Isolation | Container-level — crash/resource leak cannot affect neighbors |
| Scalability | Auto-scale workers 10 → 1,000 based on queue depth |
| Deployment | Zero-downtime worker updates (graceful drain) |

### Out of Scope (State Clearly in Interview)

- Job DAGs / workflows (chaining jobs) — future phase
- Real-time streaming jobs — different system
- Multi-region active-active — single region for now
- Billing / chargeback per team

---

## Phase 3 — High-Level Architecture

```
┌──────────────────────────────────────────────────────────────────────┐
│                          SUBMISSION LAYER                            │
│                                                                      │
│   Developer SDK  ──►  API Gateway  ──►  Job Ingestion Service       │
│                         (Rate Limit)       (Validates, Enqueues)    │
└─────────────────────────────┬────────────────────────────────────────┘
                              │ writes job metadata + enqueues message
                    ┌─────────▼──────────┐
                    │   Message Broker   │  ← SQS / RabbitMQ / Kafka
                    │  [HIGH] [NORMAL]   │    3 separate queues
                    │       [LOW]        │
                    └─────────┬──────────┘
                              │ workers poll
┌─────────────────────────────▼────────────────────────────────────────┐
│                         EXECUTION LAYER                              │
│                                                                      │
│   ┌──────────────┐    ┌──────────────┐    ┌──────────────┐         │
│   │   Worker     │    │   Worker     │    │   Worker     │  ...    │
│   │  (Pod/EC2)   │    │  (Pod/EC2)   │    │  (Pod/EC2)   │         │
│   │  Container   │    │  Container   │    │  Container   │         │
│   │  [Job A]     │    │  [Job B]     │    │  [Job C]     │         │
│   └──────────────┘    └──────────────┘    └──────────────┘         │
│           Auto-scaling Group (KEDA / ASG)                           │
└──────────────────────────┬───────────────────────────────────────────┘
                           │ writes results + status
┌──────────────────────────▼───────────────────────────────────────────┐
│                        STATE LAYER                                   │
│                                                                      │
│   Job Metadata DB (PostgreSQL)   │   Result Store (S3 / Redis)      │
│   status, retries, timestamps    │   output payloads, 7-day TTL     │
└──────────────────────────────────────────────────────────────────────┘
                           │
┌──────────────────────────▼───────────────────────────────────────────┐
│                     OBSERVABILITY LAYER                              │
│                                                                      │
│   Prometheus + Grafana  │  Structured Logs (ELK)  │  Alerts         │
└──────────────────────────────────────────────────────────────────────┘
```

---

## Phase 4 — Deep Dive: Each Component

---

### 4.1 Job Ingestion Service

**Responsibility:** Accept submissions, validate, persist metadata, enqueue.

**Why not directly enqueue from API Gateway?**
Because we need to persist job metadata to the DB *before* enqueuing. If the broker is temporarily down, the job is still saved and can be replayed. This is the **transactional outbox pattern**.

```
POST /v1/jobs
{
  "job_type": "generate_finance_report",
  "version": "v3",
  "payload": { "date": "2025-01-01", "region": "APAC" },
  "priority": "HIGH",
  "retry_policy": { "max_retries": 3, "backoff": "exponential" },
  "callback_url": "https://finance-service.internal/webhook"
}

Response 202 Accepted:
{
  "job_id": "job_01J3K...",
  "status": "QUEUED",
  "estimated_start": "~5s"
}
```

**Transactional Outbox Pattern (Critical for Durability):**

```
┌─────────────────────────────────────┐
│  Ingestion Service Transaction      │
│                                     │
│  1. INSERT job INTO jobs_table      │
│     status = QUEUED                 │
│                                     │
│  2. INSERT INTO outbox_table        │
│     (same DB transaction)           │
│                                     │
│  COMMIT (atomic)                    │
└────────────────┬────────────────────┘
                 │
         Outbox Relay Process
         (polls outbox table)
                 │
                 ▼
         Publishes to SQS/RabbitMQ
         Deletes from outbox on ACK
```

> If the broker is down, the job is safe in the DB. The outbox relay retries until the broker is available. No job is ever lost between "saved to DB" and "in the queue."

**Rate Limiting at API Gateway:**
- Per-team rate limits (e.g., 500 req/min per team)
- Global limit of 200 req/sec across all teams
- 429 Too Many Requests with `Retry-After` header

---

### 4.2 Message Broker — Queue Design

**Choice: Amazon SQS (Standard Queues) or RabbitMQ**

| Factor | SQS Standard | RabbitMQ |
|---|---|---|
| Durability | ✅ Managed, replicated | ✅ Mirrored queues |
| At-least-once | ✅ Built-in | ✅ With acks |
| Ordering | ❌ Best-effort | ✅ Per-queue FIFO |
| Ops overhead | ✅ Zero | ❌ Need to manage |
| Cost at scale | Pay-per-use | Fixed infra cost |

**Recommendation: SQS** for simplicity and durability at scale. Use SQS FIFO only for HIGH priority queue if strict ordering matters.

**Three Separate Queues:**

```
sqs://jobs-high       ← polled by HIGH workers (more workers, aggressive polling)
sqs://jobs-normal     ← polled by NORMAL workers
sqs://jobs-low        ← polled by LOW workers (can tolerate long poll intervals)

sqs://jobs-dlq        ← dead-letter queue (all priorities)
```

**Visibility Timeout — The Heartbeat Mechanism:**

When a worker picks up a message, SQS makes it invisible for `N` seconds (visibility timeout). If the worker crashes or doesn't acknowledge within `N` seconds, SQS makes the message visible again — another worker picks it up.

```
Short job (5s):  visibility timeout = 60s  (generous buffer)
Long job (2h):   visibility timeout = 7200s + heartbeat extension

Worker must call SQS ChangeMessageVisibility every 5 minutes
to extend the timeout while the job is still running.
If worker crashes → stops extending → timeout expires → requeued.
```

This is how we detect "dead/stuck" vs "still working" — **the heartbeat**.

---

### 4.3 Worker Architecture

**Design: One job = one Kubernetes Pod (container)**

```
Worker Node (EC2 instance)
├── Worker Agent (always running — the scheduler daemon)
│   ├── Polls SQS queue
│   ├── Pulls Docker image for job type
│   ├── Spawns Job Container (via Docker/CRI)
│   ├── Sends heartbeat to SQS (extends visibility timeout)
│   ├── Captures stdout/stderr
│   └── Reports result to State Layer
│
└── Job Container (ephemeral — lives only for one job)
    ├── Runs the actual job code
    ├── Has CPU/memory limits (cgroups)
    ├── Has its own network namespace
    └── Dies when job completes
```

**Resource Limits per Container (cgroups via Kubernetes):**

```yaml
resources:
  requests:
    cpu: "500m"
    memory: "512Mi"
  limits:
    cpu: "2"
    memory: "4Gi"
```

> Resource limits are declared at job type registration. The scheduler validates that a node has sufficient available resources before spawning the container.

**Worker Agent Lifecycle:**

```
1. Poll SQS queue (long polling, 20s wait)
2. Receive job message
3. Check: do I have capacity? (CPU/memory headroom)
   No → put message back (change visibility to 0), sleep 5s
4. Pull Docker image (cached locally if recently used)
5. Spawn container with resource limits
6. Start heartbeat loop (extend SQS visibility every 5min)
7. Monitor container
8. On success → write result to S3, update DB status=COMPLETED, SQS delete
9. On failure → check retry count
   retries remaining → requeue with delay, update DB status=RETRYING
   no retries left → move to DLQ, update DB status=FAILED
10. Trigger webhook callback if registered
```

---

### 4.4 Priority Queues — Preventing Head-of-Line Blocking

**Problem:** A 2-hour LOW-priority job should not block thousands of 5-second HIGH-priority jobs.

**Solution: Separate worker pools per priority tier.**

```
HIGH queue    →  Worker Pool A  (min: 20 workers, max: 500)
NORMAL queue  →  Worker Pool B  (min: 10 workers, max: 300)
LOW queue     →  Worker Pool C  (min: 2 workers, max: 100)
```

Workers in Pool A **only** poll the HIGH queue. They never touch LOW jobs. A 2-hour LOW job occupying a LOW worker has zero impact on HIGH queue latency.

**Spillover strategy (optional optimization):**
During quiet periods, HIGH workers can poll NORMAL queue if HIGH queue is empty. This improves utilization. Implement with SQS message attributes or multi-queue polling with priority ordering.

---

### 4.5 State Layer — Job Metadata Database

**PostgreSQL** (or Aurora PostgreSQL) for ACID guarantees on job state.

**Schema:**

```sql
CREATE TABLE jobs (
    job_id          UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    job_type        VARCHAR(255) NOT NULL,
    version         VARCHAR(50)  NOT NULL,
    priority        ENUM('HIGH','NORMAL','LOW') NOT NULL,
    status          ENUM('QUEUED','RUNNING','COMPLETED','FAILED','RETRYING','DEAD') NOT NULL,
    payload         JSONB,
    result_location VARCHAR(500),        -- S3 URI
    callback_url    VARCHAR(500),
    retry_count     INT DEFAULT 0,
    max_retries     INT DEFAULT 3,
    error_message   TEXT,
    worker_id       VARCHAR(255),        -- which worker is running it
    submitted_at    TIMESTAMPTZ DEFAULT NOW(),
    started_at      TIMESTAMPTZ,
    completed_at    TIMESTAMPTZ,
    expires_at      TIMESTAMPTZ          -- result retention (7 days)
);

CREATE INDEX idx_jobs_status    ON jobs(status);
CREATE INDEX idx_jobs_submitted ON jobs(submitted_at);
CREATE INDEX idx_jobs_type      ON jobs(job_type, status);
```

**Status Transitions:**

```
QUEUED → RUNNING → COMPLETED
                 ↘ RETRYING → RUNNING (retry loop)
                            ↘ DEAD (max retries exhausted)
       ↘ FAILED (non-retryable error)
```

**Result Storage:**

- Small results (< 1MB) → stored as JSONB in `jobs` table directly
- Large results (> 1MB) → stored in S3, `result_location` column holds the S3 URI
- Results expire after 7 days (S3 lifecycle policy + jobs table cleanup job)

---

### 4.6 Auto-Scaling Strategy

**Tool: KEDA (Kubernetes Event-Driven Autoscaler)**

KEDA can scale Kubernetes deployments based on external metrics — including SQS queue depth. This is the production-grade solution.

```yaml
apiVersion: keda.sh/v1alpha1
kind: ScaledObject
metadata:
  name: worker-high-scaler
spec:
  scaleTargetRef:
    name: worker-high-deployment
  minReplicaCount: 5
  maxReplicaCount: 500
  triggers:
    - type: aws-sqs-queue
      metadata:
        queueURL: https://sqs.us-east-1.amazonaws.com/xxx/jobs-high
        queueLength: "10"      # target: 10 messages per worker
        awsRegion: us-east-1
```

**Scaling Logic:**

```
Target: 10 messages per worker (queue depth / desired workers)

Queue depth = 0    → scale to minReplicaCount (5)
Queue depth = 100  → scale to 10 workers
Queue depth = 1000 → scale to 100 workers
Queue depth = 5000 → scale to 500 workers (maxReplicaCount)

Scale-up: aggressive (fast response to spikes)
Scale-down: conservative — 5-minute cooldown to avoid thrashing
```

**Node Auto-Scaling (EC2 level):**
Kubernetes Cluster Autoscaler watches for unschedulable pods (pods that can't fit on existing nodes) and provisions new EC2 instances. Workers for LOW priority run on Spot Instances (70% cost saving); HIGH priority workers run on On-Demand for reliability.

---

### 4.7 Zero-Downtime Worker Deployments

**The Problem:** You need to update worker agent code. But some workers are running 2-hour jobs. You cannot kill them.

**Solution: Graceful Drain + Rolling Update**

```
Step 1: Set worker to DRAINING mode
        Worker stops polling new jobs from SQS
        Worker continues running its current job

Step 2: Wait for current job to complete
        (up to the job's max execution time)

Step 3: Worker exits cleanly
        Kubernetes replaces it with updated version

Step 4: New worker starts polling
```

**Kubernetes Rolling Update Config:**

```yaml
strategy:
  type: RollingUpdate
  rollingUpdate:
    maxSurge: 1           # spin up 1 new worker before killing old
    maxUnavailable: 0     # never reduce capacity during update
```

**Pre-stop Hook (Graceful Drain Signal):**

```yaml
lifecycle:
  preStop:
    exec:
      command: ["/bin/sh", "-c", "touch /tmp/draining && sleep 7200"]
      # Worker's poll loop checks for /tmp/draining file
      # If present, stops polling, finishes current job, exits
```

**The Worker's Drain Check:**

```python
import os
import signal

draining = False

def handle_sigterm(signum, frame):
    global draining
    draining = True
    print("SIGTERM received — entering drain mode, finishing current job")

signal.signal(signal.SIGTERM, handle_sigterm)

while True:
    if draining:
        print("Draining — no new jobs will be picked up")
        break
    job = poll_sqs_queue()
    if job:
        execute_job(job)
```

---

### 4.8 Reliability — Retry and Dead-Letter Queue

**Retry Policy (developer-configured):**

```json
{
  "max_retries": 3,
  "backoff_strategy": "exponential",
  "base_delay_seconds": 10,
  "max_delay_seconds": 600
}
```

**Backoff Calculation:**

```
Attempt 1 fails → wait 10s  → retry
Attempt 2 fails → wait 20s  → retry
Attempt 3 fails → wait 40s  → retry
Attempt 4 fails → move to DLQ, status = DEAD
```

**Retryable vs Non-Retryable Errors:**

| Error Type | Action | Examples |
|---|---|---|
| Transient | Retry with backoff | Network timeout, rate limit, DB connection |
| Resource | Retry with delay | OOM (increase memory and retry) |
| Code bug | No retry → DLQ | `AttributeError`, `KeyError`, assertion failures |
| Timeout | Retry | Job exceeded execution time limit |

Workers classify errors by exception type. Developers can also throw a `NonRetryableError` to explicitly skip retries.

**Dead-Letter Queue:**
- All permanently failed jobs land in `sqs://jobs-dlq`
- On-call engineers get PagerDuty alerts for DLQ depth > 0
- DLQ jobs can be manually replayed via API: `POST /v1/jobs/{job_id}/replay`
- DLQ messages expire after 14 days

---

### 4.9 Observability

**The Four Golden Signals for Job Systems:**

```
1. Throughput       — jobs submitted/completed per minute per priority
2. Latency          — P50/P95/P99 time from submission to start
3. Error Rate       — job failure rate per job type
4. Queue Depth      — messages waiting per priority queue (key scaling signal)
```

**Key Metrics (Prometheus):**

```
job_submissions_total{priority, job_type}
job_completions_total{priority, job_type, status}
job_execution_duration_seconds{job_type, quantile}
job_queue_depth{priority}
job_retry_count{job_type}
worker_active_jobs{worker_id, priority}
worker_idle_count{priority}
```

**Dashboards (Grafana):**
- Real-time queue depth per priority (drives scaling awareness)
- Job throughput over time (spot submission spikes)
- P99 execution latency per job type (detect slow regressions)
- DLQ depth (alert on any non-zero value)
- Worker utilization heatmap

**Structured Logging:**

```json
{
  "timestamp": "2025-01-01T10:00:00Z",
  "level": "INFO",
  "event": "job_started",
  "job_id": "job_01J3K...",
  "job_type": "generate_finance_report",
  "worker_id": "worker-high-7f8b",
  "attempt": 1,
  "queue_wait_ms": 312
}
```

---

## Phase 5 — Handling the Hard Requirements

### Handling 10,000 Jobs/Minute (167/sec)

- API Gateway handles HTTP termination, rate limiting — scales horizontally
- Ingestion Service is stateless — scale to 10+ replicas trivially
- SQS handles 167 msg/sec effortlessly (SQS supports 300 TPS standard, unlimited with batching)
- Transactional outbox decouples API response speed from broker availability
- PostgreSQL write path is just one INSERT per submission — Aurora handles thousands of TPS

### Preventing Queue Blocking (Short vs Long Jobs)

- Separate worker pools per priority — long LOW jobs never block HIGH workers
- SHORT jobs get HIGH priority — they start within seconds
- LONG jobs get NORMAL/LOW priority — they wait for appropriate pool availability
- Execution time limits per job type — prevents infinite running jobs from permanently occupying workers

### Handling 2-Hour Jobs (Visibility Timeout + Heartbeat)

```python
# Worker heartbeat — runs in separate thread
def heartbeat_loop(sqs_client, receipt_handle, interval=300):
    while job_running:
        sqs_client.change_message_visibility(
            QueueUrl=QUEUE_URL,
            ReceiptHandle=receipt_handle,
            VisibilityTimeout=600  # extend by 10 more minutes
        )
        time.sleep(interval)
```

If worker crashes → heartbeat stops → visibility timeout expires → job requeued → new worker picks it up.

---

## Phase 6 — Trade-offs and Alternatives

### Trade-off 1: SQS vs Kafka

| | SQS | Kafka |
|---|---|---|
| Use case | Task queue (each message consumed once) | Event stream (messages replayed) |
| Ops overhead | Zero (managed) | High (need to manage brokers, partitions) |
| Replay | Via DLQ replay API | Native log replay |
| At-least-once | ✅ | ✅ |
| Ordering | Best-effort (FIFO queue for strict) | Per-partition ordering |
| **Verdict** | ✅ Better for job queues | Overkill unless you need event sourcing |

### Trade-off 2: Container-per-Job vs Process-per-Job

| | Container | Process |
|---|---|---|
| Isolation | Strong (namespace, cgroups) | Weak (shared interpreter) |
| Startup overhead | 1-5s (image pull cached) | < 100ms |
| Dependency conflicts | ✅ Solved | ❌ Still a problem |
| Resource limits | ✅ Hard limits via cgroups | ❌ Best effort |
| **Verdict** | ✅ Required for multi-tenant | Use only for trusted, homogeneous jobs |

### Trade-off 3: At-Least-Once vs Exactly-Once

- **Exactly-once** requires distributed transactions — complex, slow, expensive
- **At-least-once** with idempotent jobs is the industry standard
- We ensure idempotency by: jobs checking if their output already exists before writing, using `job_id` as an idempotency key in all downstream writes

### Trade-off 4: Pull vs Push Worker Model

| | Pull (Workers poll SQS) | Push (Broker pushes to workers) |
|---|---|---|
| Back-pressure | ✅ Natural — worker only pulls when ready | ❌ Broker pushes regardless of worker load |
| Complexity | ✅ Simple | ❌ Broker needs worker registry |
| **Verdict** | ✅ Pull model is standard for job queues | |

---

## Phase 7 — Final Architecture Summary Diagram

```
Developers
    │
    ▼
[Python SDK]
    │
    ▼
[API Gateway]  ←─ rate limiting, auth
    │
    ▼
[Job Ingestion Service] (stateless, horizontally scaled)
    │
    ├─── Writes job to PostgreSQL (status=QUEUED)
    └─── Writes to Outbox table (same transaction)
              │
         [Outbox Relay Process]
              │
              ▼
    ┌─────────────────────┐
    │    SQS Queues        │
    │  [HIGH]             │
    │  [NORMAL]           │
    │  [LOW]              │
    │  [DLQ]              │
    └──────────┬──────────┘
               │ workers poll
    ┌──────────▼──────────────────────────────────┐
    │         Worker Pools (Kubernetes)            │
    │                                              │
    │  Pool-HIGH  →  Job Container (isolated)      │
    │  Pool-NORMAL → Job Container (isolated)      │
    │  Pool-LOW   →  Job Container (isolated)      │
    │                                              │
    │  KEDA autoscales each pool on queue depth    │
    └──────────┬──────────────────────────────────┘
               │
    ┌──────────▼──────────────────────────────────┐
    │           State Layer                        │
    │  PostgreSQL — job status, metadata           │
    │  S3 — job results (7-day TTL)                │
    │  Redis — hot status cache (optional)         │
    └──────────┬──────────────────────────────────┘
               │
    ┌──────────▼──────────────────────────────────┐
    │        Observability                         │
    │  Prometheus + Grafana (metrics)              │
    │  ELK Stack (logs)                            │
    │  PagerDuty (DLQ alerts)                      │
    └─────────────────────────────────────────────┘
```

---

## Phase 8 — Interview-Ready One-Paragraph Summary

> "The system has three layers. The submission layer uses a stateless Ingestion Service behind an API Gateway — it persists each job to PostgreSQL and publishes to SQS using the transactional outbox pattern, guaranteeing durability even if the broker is temporarily down. The execution layer uses Kubernetes worker pools — one pool per priority tier — where each job runs in an isolated container with hard resource limits. KEDA autoscales each pool based on SQS queue depth. Workers use SQS visibility timeout as a heartbeat mechanism — if a worker crashes, the timeout expires and the job is automatically requeued. The state layer tracks all job metadata in PostgreSQL and stores results in S3 with a 7-day TTL. Zero-downtime deployments are handled by a SIGTERM drain handler — workers stop polling when they receive a shutdown signal, finish their current job, then exit cleanly. The key trade-offs are: at-least-once over exactly-once for simplicity, container isolation over process isolation for safety, and pull-based workers over push for natural backpressure."

---

## Quick-Reference: Key Design Decisions

| Problem | Solution | Why |
|---|---|---|
| 10K submissions/min | SQS + stateless ingestion service | SQS handles unlimited throughput, service scales horizontally |
| Durability | Transactional outbox pattern | Job saved to DB and broker atomically — never lost |
| Long job detection | SQS visibility timeout + heartbeat | Worker extends timeout while alive; crash = requeue |
| Head-of-line blocking | Separate worker pools per priority | Long LOW jobs never delay HIGH short jobs |
| Dependency conflicts | Container-per-job | Each job gets isolated filesystem and network |
| Resource fairness | Kubernetes resource limits (cgroups) | One runaway job cannot starve others |
| Auto-scaling | KEDA on SQS queue depth | Scale workers proportional to actual work waiting |
| Zero-downtime deploy | Graceful drain via SIGTERM | Workers finish current job before accepting update |
| Permanent failures | Dead-letter queue + alerts | Nothing silently lost; ops team investigates |
| Exactly-once semantics | Idempotent jobs + job_id key | At-least-once is sufficient when jobs are idempotent |